# RelFuse-Net — Real training on Colab

Pulls the current code from GitHub (`minhhaidhsp/RelFuse-Net`), copies the real preprocessed MIMIC data from Drive, installs dependencies, and runs `train.py --ablation` (Scenario A + B). Epoch count and number of seeds are configurable in the **Run config** cell below -- the repo's real defaults are `EPOCHS=50`, `NUM_RUNS=5`, but you can lower these to fit your Colab compute-unit budget.

**Colab and the Drive data now run under the same Google account (`minhhaidhsp@gmail.com`)** -- no more cross-account sharing/shortcut step needed. The mount cell below points straight at `My Drive/NCKH/Paper/RelFuse/RelFuse-Net-colab` and only falls back to a (slower) search of the whole Drive if that exact path isn't found, e.g. if you moved the folder.

**Re-running without re-copying images:** the clone cell and the data-copy cell are both safe to re-run on a runtime that's still alive (e.g. after bumping `RUN_EPOCHS` and doing Run All again, or after a brief hiccup that didn't actually kill the runtime) -- the clone cell updates the code in place instead of wiping `/content/RelFuse-Net`, and the data-copy cell skips any file already present locally with the right size. Only a genuinely new/recycled Colab runtime (local disk actually empty) forces a full re-copy -- there's no way around that. Either way, training progress itself is never lost to a disconnect: see "Checkpoint durability" below.

**If the clone cell errors out (e.g. after a network hiccup during an earlier clone left a partial checkout):** it now removes any broken partial checkout and re-clones automatically, and raises a clear error immediately if git itself fails, instead of leaving you with a confusing "config.py not found" a few cells later. Just re-run the clone cell (or Run All again) if you see an error there.

**Progress bars:** the dependency-install, data-copy, and training cells all show a live progress bar (with an ETA once it has a few data points to estimate from) instead of sitting silently -- the training cell in particular shows one overall bar covering every epoch across all seeds/scenarios, plus a per-batch bar for whichever epoch is currently running.

**Runtime & budget:** Settings → Change runtime type → GPU. Try **L4** first (cheaper in compute units than A100, and should have enough VRAM for the 8B LLM with 4-bit quantization + LoRA) -- only switch to A100 if you hit an out-of-memory error. If your remaining compute-unit budget is limited, do a short probe run first (set `RUN_EPOCHS = 3`, `RUN_NUM_RUNS = 1` in the Run config cell, run the whole notebook once, and check the unit counter in the top-right of the Colab UI) before committing to a full run.

**If your session disconnects partway through:** just reopen this notebook and Run All again from the top. The "Restore checkpoint from Drive" cell automatically copies back any resume-state the background-sync cell saved, and `train.py` picks up from the last completed epoch on its own -- no manual file copying needed. (The overall progress bar also picks up correctly: epochs already completed before the disconnect count toward it immediately, so its ETA doesn't reset to zero.)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Colab and Drive are the same account now, so we know the exact path --
# no need to search the whole Drive tree for it. (The previous version used
# glob('**', recursive=True) from the My Drive root to auto-find the folder
# across two different Google accounts; on an account with a lot of files
# that recursive walk over the Drive FUSE mount can hang for a very long
# time -- that's what you just interrupted with Ctrl-C / stop button.)
DRIVE_ROOT = '/content/drive/MyDrive/NCKH/Paper/RelFuse/RelFuse-Net-colab'

if not os.path.isdir(DRIVE_ROOT):
    # Fallback only if you moved the folder -- this recursive search can be
    # slow on a large Drive, so it only runs when the direct path above
    # doesn't exist.
    import glob
    print(f'{DRIVE_ROOT} not found -- falling back to a slower search under '
          f'My Drive (this can take a while on a large Drive, please wait)...')
    _candidates = [p for p in glob.glob('/content/drive/MyDrive/**/RelFuse-Net-colab', recursive=True) if os.path.isdir(p)]
    assert _candidates, (
        "Could not find a 'RelFuse-Net-colab' folder under My Drive. Check "
        "that it still exists at NCKH/Paper/RelFuse/RelFuse-Net-colab, or "
        "edit DRIVE_ROOT above to point at it directly."
    )
    DRIVE_ROOT = _candidates[0]

print('Drive folder OK:', DRIVE_ROOT)


In [ ]:
import os, shutil, subprocess

# Clone only if this runtime doesn't already have a WORKING repo checkout --
# reusing an existing checkout (fetch + hard reset instead of rm -rf +
# re-clone) means data/ (gitignored, so git reset never touches it) survives
# a re-run of this cell on the SAME still-alive runtime. So bumping
# RUN_EPOCHS and re-running the notebook, or just re-running everything after
# a brief hiccup that didn't actually kill the runtime, does NOT force you to
# re-copy images from Drive. A brand-new/recycled Colab runtime has no
# /content/RelFuse-Net yet, so it still clones fresh exactly as before.
#
# Hardened: every git step's exit code is checked and raises immediately on
# failure instead of silently continuing into a confusing error several
# cells later (e.g. "config.py not found"). If /content/RelFuse-Net exists
# but ISN'T a valid checkout (no .git -- e.g. a previous clone was
# interrupted by a network hiccup and left a partial/empty directory), it is
# removed first so `git clone` doesn't refuse with "destination path already
# exists and is not empty".

REPO_DIR = '/content/RelFuse-Net'
REPO_URL = 'https://github.com/minhhaidhsp/RelFuse-Net.git'


def _run(cmd, cwd=None):
    print('+', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=cwd)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit code {result.returncode}): {' '.join(cmd)}\n"
            f"Check the git output printed above for the actual error."
        )


if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('Repo already checked out on this runtime -- updating code in place '
          '(data/ is left untouched).')
    _run(['git', 'fetch', 'origin', 'main'], cwd=REPO_DIR)
    _run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO_DIR)
else:
    if os.path.isdir(REPO_DIR):
        print(f'{REPO_DIR} exists but is not a valid git checkout (likely a '
              f'partial/interrupted clone from earlier) -- removing it before '
              f'cloning fresh.')
        shutil.rmtree(REPO_DIR)
    print('No existing checkout on this runtime -- cloning fresh.')
    _run(['git', 'clone', REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
_run(['git', 'log', '--oneline', '-3'], cwd=REPO_DIR)

# Sanity check: config.py must exist now, or every later cell fails with a
# confusing "file not found" instead of a clear error at the actual source.
assert os.path.isfile(os.path.join(REPO_DIR, 'config.py')), (
    f"{REPO_DIR}/config.py is missing after clone/update -- the git operation "
    f"above did not actually produce a working checkout. Check the output "
    f"above for the real git error and re-run this cell."
)
print('Repo ready at', REPO_DIR)


In [ ]:
# ---- Run config: chinh 2 so duoi day tuy ngan sach compute unit Colab cua ban ----
# Mac dinh goc cua repo la EPOCHS=50, NUM_RUNS=5 (5 seed doc lap x 2 scenario).
# Ha EPOCHS xuong an toan ve mat code (lich giam LR cosine-annealing tu bam theo
# dung so EPOCHS nay), chi anh huong muc do hoi tu cua model. Ha NUM_RUNS lam
# giam do tin cay thong ke (mean +/- std o Bang 7/8) nhung tung seed van cho ket
# qua hop le -- co the chay bo sung seed sau neu con ngan sach.
RUN_EPOCHS = 3
RUN_NUM_RUNS = 1
# --------------------------------------------------------------------------------

import re

cfg_path = '/content/RelFuse-Net/config.py'
cfg = open(cfg_path).read()

cfg, n_epochs = re.subn(r'(?m)^(\s*)EPOCHS = \d+', rf'\1EPOCHS = {RUN_EPOCHS}', cfg)
cfg, n_runs = re.subn(r'(?m)^(\s*)NUM_RUNS = \d+', rf'\1NUM_RUNS = {RUN_NUM_RUNS}', cfg)
assert n_epochs == 1 and n_runs == 1, (
    f"Expected exactly 1 match each in config.py, got EPOCHS matches={n_epochs} "
    f"NUM_RUNS matches={n_runs} -- config.py's layout may have changed upstream, "
    f"edit EPOCHS/NUM_RUNS in config.py by hand instead."
)
open(cfg_path, 'w').write(cfg)
print(f'config.py patched: EPOCHS={RUN_EPOCHS}, NUM_RUNS={RUN_NUM_RUNS} '
      f'(repo defaults were EPOCHS=50, NUM_RUNS=5)')


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# torch/torchvision already come preinstalled (correct CUDA build) on Colab --
# do NOT reinstall them. Add torch_geometric + its neighbor-sampling extras
# (pyg-lib or torch-sparse -- NeighborLoader requires one of the two), matched
# to the exact torch/CUDA build this runtime has.
import time
_deps_t0 = time.time()

import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
print('torch', TORCH_VERSION, '/', CUDA_VERSION)

# No -q here on purpose: pip's own per-package download/progress output is the
# only feedback you get during this cell, so it's left visible instead of
# silenced -- otherwise this can sit for a few minutes with no sign of life.
!pip install torch_geometric
# Prebuilt wheels first (fast); if the PyG wheel index doesn't have this exact
# torch/CUDA combo yet, fall back to building torch-scatter/torch-sparse from
# source via plain PyPI (slower, ~a few minutes, but always works).
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_VERSION}.html \
  || pip install torch_scatter torch_sparse

!pip install transformers peft accelerate bitsandbytes scikit-learn pillow tqdm pandas

import torch_geometric, torch_geometric.typing as tgt
print('torch_geometric', torch_geometric.__version__,
      '| WITH_PYG_LIB', tgt.WITH_PYG_LIB, '| WITH_TORCH_SPARSE', tgt.WITH_TORCH_SPARSE)
assert tgt.WITH_PYG_LIB or tgt.WITH_TORCH_SPARSE, (
    'Neither pyg-lib nor torch-sparse installed -- NeighborLoader will fail. '
    'Re-run the cell above, or install torch_scatter/torch_sparse manually for this torch/CUDA version.'
)
print(f'Dependencies installed in {time.time() - _deps_t0:.0f}s.')


In [ ]:
# Copy the real preprocessed data from Drive to local disk (fast reads during
# training -- Drive's FUSE mount is too slow for thousands of small per-batch
# image reads per epoch). This is a one-time copy per Colab session.
#
# Skips any file that's already present locally with the right size, so
# re-running this cell (e.g. as part of Run All again after bumping RUN_EPOCHS,
# or after a brief disconnect that didn't actually recycle the runtime) is fast
# instead of re-copying all ~7000 images from scratch every time. On a genuinely
# NEW/recycled runtime (local disk really is empty) it still copies everything,
# same as before -- there's no way around that: the files have to physically
# exist on whichever VM is running right now.
import os, shutil
from tqdm.auto import tqdm

src = os.path.join(DRIVE_ROOT, 'data')
dst = '/content/RelFuse-Net/data'
assert os.path.isdir(src), (
    f"{src} not found -- upload the data/ folder (data/processed/ + "
    f"data/mimic_subset/mimic-cxr-jpg/2.1.0/files/) into the minhhaidhsp@gmail.com "
    f"copy of RelFuse-Net-colab/data/ first (see the intro cell)."
)

print('Scanning source folder on Drive (this part has no progress bar -- it is '
      'just listing directories over the Drive mount, usually well under a minute)...')
all_files = []
for root, _dirs, files in os.walk(src):
    for fn in files:
        all_files.append(os.path.join(root, fn))
print(f'{len(all_files)} files in source.')

to_copy = []
for f in all_files:
    rel = os.path.relpath(f, src)
    out_path = os.path.join(dst, rel)
    if os.path.exists(out_path) and os.path.getsize(out_path) == os.path.getsize(f):
        continue  # already copied and looks intact -- skip
    to_copy.append((f, out_path))

if len(to_copy) < len(all_files):
    print(f'{len(all_files) - len(to_copy)} files already present locally with the right size -- skipping those.')
print(f'{len(to_copy)} files left to actually copy.')

for f, out_path in tqdm(to_copy, desc='Copying data/ from Drive to local disk', unit='file'):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    shutil.copy2(f, out_path)

for f in ['mimic_train.csv', 'mimic_val.csv', 'mimic_test.csv', 'train_graph_edges.pt']:
    p = f'/content/RelFuse-Net/data/processed/{f}'
    print(f, 'OK' if os.path.exists(p) else 'MISSING')
n_imgs = sum(len(files) for _, _, files in os.walk('/content/RelFuse-Net/data/mimic_subset/mimic-cxr-jpg'))
print('image files found:', n_imgs)


In [ ]:
# Tu dong khoi phuc checkpoint/resume-state tu lan chay truoc (neu Colab tung bi
# ngat giua chung) truoc khi bat dau training. Neu day la lan chay dau tien, se
# khong tim thay gi va train.py bat dau tu epoch 0 nhu binh thuong.
import os, glob, shutil

DRIVE_CKPT_DIR = os.path.join(DRIVE_ROOT, 'checkpoints_live')
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

_patterns = ['relfusenet_resume_*.pt', 'relfusenet_best_*.pth', 'relfusenet_results_*.json']
_restored = []
for _pattern in _patterns:
    for _f in glob.glob(os.path.join(DRIVE_CKPT_DIR, _pattern)):
        _dst = os.path.join('/content/RelFuse-Net', os.path.basename(_f))
        shutil.copy2(_f, _dst)
        _restored.append(os.path.basename(_f))

if _restored:
    print('Tim thay checkpoint tu phien truoc, da khoi phuc ve local:')
    for _r in sorted(_restored):
        print(' -', _r)
    print('train.py se tu resume tu epoch/seed/scenario da luu, khong chay lai tu dau.')
else:
    print('Khong tim thay checkpoint cu tren Drive -- day la lan chay dau tien, se train tu dau.')


## Checkpoint durability

`train.py` already saves a full resume-state (model/optimizer/scheduler/vCLUB nets/RNG) every epoch and the best-val checkpoint whenever it improves, both as local files in the repo's working directory. The cell above already restored any such files from Drive automatically, and the background-sync cell below copies them back to Drive every 5 minutes while training runs. **So if Colab disconnects, you don't need to do anything by hand: just reopen this notebook and Run All from the top again** -- the restore cell picks up where the last session left off, and `train.py`'s own resume logic continues from the last completed epoch instead of restarting.


In [ ]:
import threading, time, glob, shutil, os

DRIVE_CKPT_DIR = os.path.join(DRIVE_ROOT, 'checkpoints_live')
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

_stop_sync = threading.Event()

def _sync_loop():
    patterns = ['relfusenet_best_*.pth', 'relfusenet_resume_*.pt', 'relfusenet_results_*.json']
    while not _stop_sync.is_set():
        for pattern in patterns:
            for f in glob.glob(f'/content/RelFuse-Net/{pattern}'):
                try:
                    shutil.copy2(f, DRIVE_CKPT_DIR)
                except Exception as e:
                    print('[drive-sync] warning:', e)
        _stop_sync.wait(300)  # every 5 minutes

_sync_thread = threading.Thread(target=_sync_loop, daemon=True)
_sync_thread.start()
print('Background Drive sync started (every 5 min) ->', DRIVE_CKPT_DIR)

In [ ]:
# Real run: Config.USE_REAL_LLM=True (repo default, unmodified). EPOCHS/NUM_RUNS
# come from the Run config cell above (RUN_EPOCHS/RUN_NUM_RUNS), not necessarily
# the repo's original 50/5. --ablation trains + evaluates both Scenario A
# (report-free, prospective) and Scenario B (full model) and writes
# relfusenet_results_scenarioA.json / ...scenarioB.json.
#
# Shows a live "Overall training" progress bar (all epochs, both scenarios,
# all seeds) with an ETA, plus a per-batch bar for the epoch currently running.
%cd /content/RelFuse-Net
!python train.py --ablation


In [ ]:
# Final sync once training finishes cleanly (the background thread already
# covers interruptions, this just guarantees everything is on Drive at the end).
_stop_sync.set()
import glob, shutil
for pattern in ['relfusenet_best_*.pth', 'relfusenet_results_*.json']:
    for f in glob.glob(f'/content/RelFuse-Net/{pattern}'):
        shutil.copy2(f, DRIVE_CKPT_DIR)
print('Final checkpoints/results synced to', DRIVE_CKPT_DIR)

import json, glob
for f in sorted(glob.glob('/content/RelFuse-Net/relfusenet_results_*.json')):
    print('\n===', f, '===')
    print(json.dumps(json.load(open(f)), indent=2)[:2000])